# Model Versioning And Information
This notebook is intended to track the model versions. This includes metrics on model performance, hyperparameters, algorithms used, and any additional information related to the training model. Due to limited resources in AWS, the models are not trained within AWS and are instead trained on local hardward. The information for the Model groups are generated from uploaded h5 files. 

In [2]:
import boto3
import os
import sagemaker
import pandas as pd 
import numpy as np 
import tensorflow as tf
import json
from sagemaker.model import Model

In [11]:
!pip install -U sagemaker

  Using cached sagemaker_core-1.0.22-py3-none-any.whl.metadata (4.9 kB)
  Using cached s3transfer-0.11.2-py3-none-any.whl.metadata (1.7 kB)
  Using cached mock-4.0.3-py3-none-any.whl.metadata (2.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.1 MB/s eta 0:00:00
Using cached sagemaker_core-1.0.22-py3-none-any.whl (404 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 115.7 MB/s eta 0:00:00
Using cached mock-4.0.3-py3-none-any.whl (28 kB)
Using cached s3transfer-0.11.2-py3-none-any.whl (84 kB)
  Attempting uninstall: mock
    Found existing installation: mock 5.1.0
    Uninstalling mock-5.1.0:
      Successfully uninstalled mock-5.1.0
  Attempting uninstall: botocore
    Found existing installation: botocore 1.34.162
    Uninstalling botocore-1.34.162:
      Successfully uninstalled botocore-1.34.162
  Attempting uninstall: s3transfer
    Found existing installation: s3transfer 0.10.4
    Uninstalling s3transfer-0.10.4:
      Successfully uninstalled s3transfer

In [6]:
sagemaker_session = sagemaker.Session()
bucket = sagemaker_session.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
account_id = boto3.client("sts").get_caller_identity().get("Account")
 
s3_client = boto3.client(service_name="s3", region_name=region)
sagemaker_client = boto3.client("sagemaker", region_name=region)
model_package_group_name = "FER2013-Model-Group"

# Define S3 path for the model
model_s3_path = f"s3://{bucket}/models/model.keras"

In [4]:
# Create Model Package Group (if not exists)
try:
    sagemaker_client.create_model_package_group(
        ModelPackageGroupName=model_package_group_name,
        ModelPackageGroupDescription="Model Group for FER2013 Emotion Detection"
    )
    print(f"Model package group '{model_package_group_name}' created.")
except sagemaker_client.exceptions.ResourceInUseException:
    print(f"Model package group '{model_package_group_name}' already exists.")

Model package group 'FER2013-Model-Group' created.


In [7]:
# Upload model to S3
s3_client.upload_file("fer_best_model.keras", bucket, "models/model.keras")
print(f"Model uploaded to {model_s3_path}")

Model uploaded to s3://sagemaker-us-east-1-399018723364/models/model.keras


In [8]:
# Load model to extract metadata
model = tf.keras.models.load_model("fer_best_model.keras")
config = model.get_config()
optimizer_config = model.optimizer.get_config()

In [9]:
# Extract hyperparameters and model details
model_metadata = {
    "Algorithm": "Convolutional Neural Network (CNN)",
    "Framework": "TensorFlow",
    "Hyperparameters": {
        "optimizer": model.optimizer.__class__.__name__,
        "learning_rate": optimizer_config.get("learning_rate", "Unknown"),
        "batch_size": "Unknown",  # Batch size is usually set during training
        "epochs": "Unknown"  # Epochs are not stored in the model file
    },
    "Architecture": [layer["class_name"] for layer in config["layers"]],
    "Loss Function": model.loss,
}


In [16]:
# Register Model Package
response = sagemaker_client.create_model_package(
    ModelPackageGroupName=model_package_group_name,
    ModelPackageDescription="FER2013 Emotion Detection Model",
    InferenceSpecification={
        "Containers": [
            {
                "Image": sagemaker.image_uris.retrieve(
                    framework="tensorflow",
                    region=region,
                    version="2.8",
                    instance_type="ml.m5.large",
                    image_scope="inference"
                ),
                "ModelDataUrl": model_s3_path,
            }
        ],
        "SupportedContentTypes": ["application/x-npy"],
        "SupportedResponseMIMETypes": ["application/json"],
    },
    MetadataProperties={
        "GeneratedBy": "SageMaker",
        "ProjectId": "FER2013",
    }
)

# Get Model Package Group Arn
model_package_group_arn = f"arn:aws:sagemaker:{region}:{account_id}:model-package-group/{model_package_group_name}"

# Ensure tags comply with AWS constraints
def sanitize_tag_value(value):
    return value.replace(" ", "_").replace("(", "").replace(")", "").replace(",", "_")[:256]  # Remove invalid characters and truncate

sagemaker_client.add_tags(
    ResourceArn=model_package_group_arn,
    Tags=[
        {"Key": "Algorithm", "Value": "Convolutional_Neural_Network_CNN"},
        {"Key": "Framework", "Value": "TensorFlow"},
        {"Key": "Optimizer", "Value": sanitize_tag_value(model.optimizer.__class__.__name__)},
        {"Key": "LearningRate", "Value": str(optimizer_config.get("learning_rate", "Unknown"))},
        {"Key": "Architecture", "Value": sanitize_tag_value(", ".join([layer["class_name"] for layer in config["layers"]]))},
        {"Key": "LossFunction", "Value": sanitize_tag_value(str(model.loss))}
    ]
)


print("Model registered successfully:", model_package_group_arn)

Model registered successfully: arn:aws:sagemaker:us-east-1:399018723364:model-package-group/FER2013-Model-Group


In [ ]:
# Deploy Model as Batch Transform Job
transformer = model.transformer(
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket}/batch-predictions/"
)

transformer.transform(
    data=f"s3://{bucket}/test/test/",
    content_type="application/x-npy",
    split_type="Line"
)


## Release Resources

In [20]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>

In [1]:
%%javascript

try {
    Jupyter.notebook.save_checkpoint();
    Jupyter.notebook.session.delete();
}
catch(err) {
    // NoOp
}

<IPython.core.display.Javascript object>